# Med3D ➜ TabPFN (Step-by-step, then all-at-once)

This notebook reflects the current pipeline behavior:
- Folder-level validation split is used for caching/extraction only.
- Evaluation uses a feature-level STRATIFIED split from the union of features (train+val).
- A single-method orchestrator is provided for multi-dataset runs.


In [ ]:
# Environment & threading caps to reduce OpenMP conflicts and oversubscription
import os, sys, site, torch
os.environ['PYTHONNOUSERSITE'] = '1'
try:
    usr = site.getusersitepackages()
    sys.path = [p for p in sys.path if p != usr]
except Exception:
    pass
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
os.environ['MKL_THREADING_LAYER'] = 'SEQUENTIAL'
try:
    torch.set_num_threads(1)
    torch.set_num_interop_threads(1)
except Exception:
    pass
print('Thread caps applied.')

In [ ]:
# Locate repo root so imports and relative paths work regardless of launch dir
from pathlib import Path
def _add_repo_root_to_sys_path():
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        if (base / 'med3pipe').is_dir():
            if str(base) not in sys.path:
                sys.path.insert(0, str(base))
            return base
    raise RuntimeError("Could not locate 'med3pipe/' in current or parent directories.")
repo_root = _add_repo_root_to_sys_path()
print('Repo root:', repo_root)
config_path = repo_root / 'configs' / 'datasets.yaml'
outputs_base = repo_root / 'notebooks'
print('Config path:', config_path)
print('Outputs base:', outputs_base)

## Steps 1–2: Prepare dataset for SAM-Med3D

In [ ]:
from med3pipe.data import prepare_for_sam3d, find_default_sam3d_root
SAM3D_ROOT = find_default_sam3d_root()
CATEGORY = 'gist'
CT_NAME = 'ct_GIST'
DATASET_ROOT = repo_root / 'gist'
prepared, paths = prepare_for_sam3d(
    dataset_root=DATASET_ROOT,
    sam3d_root=SAM3D_ROOT,
    category=CATEGORY,
    ct_name=CT_NAME,
    case_glob=None,
    max_cases=None,
)
print('Prepared cases:', prepared)
print('Train images folder:', paths.images_tr)
print('Train labels folder:', paths.labels_tr)

## Step 3: Create folder-level validation split (for caching only)

In [ ]:
from med3pipe.data import split_validation
n_tr, n_val = split_validation(paths, split_ratio=0.8, seed=2025, copy=True)
print('Split done | Train:', n_tr, '| Val:', n_val)
print('Val images folder:', paths.images_val)
print('Val labels folder:', paths.labels_val)

## Step 4: Build SAM-Med3D encoder and extract embeddings

In [ ]:
import torch
# Try to find checkpoint
checkpoint_path = SAM3D_ROOT / 'ckpt' / 'sam_med3d_turbo.pth'
if not checkpoint_path.exists():
    checkpoint_path = SAM3D_ROOT / 'ckpt' / 'SAM-Med3D-turbo.pth'
if not checkpoint_path.exists():
    print("⚠️  WARNING: No checkpoint found! Model will use random weights.")
    print(f"   Download from: https://huggingface.co/blueyo0/SAM-Med3D/blob/main/sam_med3d_turbo.pth")
    print(f"   Save to: {SAM3D_ROOT / 'ckpt' / 'sam_med3d_turbo.pth'}")
    checkpoint_path = None
else:
    print(f"✅ Using checkpoint: {checkpoint_path}")
from med3pipe.sam.core import build_sam3d_model, default_feature_dirs, extract_embeddings_train_val
# Try to find checkpoint
checkpoint_path = SAM3D_ROOT / 'ckpt' / 'sam_med3d_turbo.pth'
if not checkpoint_path.exists():
    checkpoint_path = SAM3D_ROOT / 'ckpt' / 'SAM-Med3D-turbo.pth'
if not checkpoint_path.exists():
    print("⚠️  WARNING: No checkpoint found! Model will use random weights.")
    print(f"   Download from: https://huggingface.co/blueyo0/SAM-Med3D/blob/main/sam_med3d_turbo.pth")
    print(f"   Save to: {SAM3D_ROOT / 'ckpt' / 'sam_med3d_turbo.pth'}")
    checkpoint_path = None
else:
    print(f"✅ Using checkpoint: {checkpoint_path}")
model = build_sam3d_model(sam3d_root=SAM3D_ROOT, model_type='vit_b_ori', checkpoint=checkpoint_path)
feat_dirs = default_feature_dirs(SAM3D_ROOT, category=paths.category, ct_name=paths.ct_name)
extract_embeddings_train_val(paths, model, sam3d_root=SAM3D_ROOT, img_size=128, feature_dirs=feat_dirs)
feat_dirs

## Step 5–6: ROI-pooled features and labels

In [ ]:
from med3pipe.sam.core import load_labels_from_sheet
sheet_csv = repo_root / 'gist' / 'sheet.csv'
df, lab_map = load_labels_from_sheet(
    sheet_csv=sheet_csv, dataset_name='GIST', subject_col='Subject', label_col='Diagnosis_binary', case_suffix='_CT'
)
print('Labels loaded:', len(df))
print(df.head(3))

## Step 6b: Feature-level STRATIFIED split from union of features

In [ ]:
from med3pipe.tabular.stratify import stratified_features_split
(X_train, y_train, ids_train), (X_val, y_val, ids_val) = stratified_features_split(
    feat_train_dir=feat_dirs.train_dir,
    feat_val_dir=feat_dirs.val_dir,
    labels_tr_dir=paths.labels_tr,
    labels_val_dir=paths.labels_val,
    lab_map=lab_map,
    train_ratio=0.8,
    seed=2025,
)
X_train.shape, X_val.shape, len(ids_train), len(ids_val)

## Step 7: Standardize + PCA (fit on TRAIN only)

In [ ]:
from med3pipe.tabular import standardize_pca
X_train_p, X_val_p, scaler, pca = standardize_pca(X_train=X_train, X_val=X_val, n_components_max=500, random_state=42)
X_train_p.shape, X_val_p.shape

## Step 8: Train/evaluate TabPFN

In [ ]:
from med3pipe.tabular import tabpfn_pipeline
res = tabpfn_pipeline(
    X_train=X_train, y_train=y_train,
    X_val=X_val, y_val=y_val, ids_val=ids_val,
    category=paths.category, ct_name=paths.ct_name,
)
print('Saved outputs under:', res['out_dir'])
print('Metrics:', res.get('metrics'))

# All at once: TabPFN across datasets from YAML (single-method)

In [ ]:
from med3pipe.pipelines import run_multi_tabpfn
res_tab = run_multi_tabpfn(config_path=config_path, outputs_base_dir=outputs_base, dataset_names=('gist',))
res_tab['summary_df']